# DeepEval MCP Metrics Notebook

This notebook evaluates hardcoded GitHub MCP-style interactions with DeepEval's MCP metrics.

It runs 9 evaluations in total:

- `MCPUseMetric`: 2 positive + 1 negative
- `MultiTurnMCPUseMetric`: 2 positive + 1 negative
- `MCPTaskCompletionMetric`: 2 positive + 1 negative

Results are shown metric-wise instead of as one combined table.

## 1. Install Dependencies

In [ ]:
%pip install deepeval mcp openai pandas

## 2. Imports

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable

import pandas as pd

from deepeval.metrics.mcp.mcp_task_completion import MCPTaskCompletionMetric
from deepeval.metrics.mcp.multi_turn_mcp_use_metric import MultiTurnMCPUseMetric
from deepeval.metrics.mcp_use_metric.mcp_use_metric import MCPUseMetric
from deepeval.models.llms.azure_model import AzureOpenAIModel
from deepeval.test_case import ConversationalTestCase, LLMTestCase, MCPServer, MCPToolCall, Turn
from mcp.types import CallToolResult, TextContent, Tool

## 3. Credentials

Paste your real values below before running the evaluation cells.

Important: `AZURE_OPENAI_ENDPOINT` should be only the Azure resource endpoint, for example `https://your-resource.openai.azure.com`. Do not paste the full `/chat/completions?...` URL.

In [ ]:
AZURE_OPENAI_API_KEY = "PASTE_AZURE_OPENAI_API_KEY_HERE"
AZURE_OPENAI_ENDPOINT = "https://PASTE_RESOURCE_NAME.openai.azure.com"
AZURE_OPENAI_DEPLOYMENT_NAME = "PASTE_AZURE_OPENAI_DEPLOYMENT_NAME_HERE"
AZURE_OPENAI_MODEL_NAME = "gpt-5"
AZURE_OPENAI_API_VERSION = "PASTE_AZURE_OPENAI_API_VERSION_HERE"

# Included for future real GitHub MCP calls. The hardcoded test cases below do not call GitHub.
GITHUB_PAT = "PASTE_GITHUB_PERSONAL_ACCESS_TOKEN_HERE"

# Set these only if DeepEval does not recognize your model pricing yet.
# Use per-token costs, not per-1K or per-1M token costs.
AZURE_OPENAI_COST_PER_INPUT_TOKEN = 0.0
AZURE_OPENAI_COST_PER_OUTPUT_TOKEN = 0.0

## 4. Hardcoded GitHub MCP Settings

In [ ]:
GITHUB_OWNER = "octocat"
GITHUB_REPO = "Hello-World"
GITHUB_ISSUE_NUMBER = 348

## 5. Result Schema and Judge Model

In [ ]:
@dataclass(frozen=True)
class MetricResult:
    case_id: str
    metric: str
    expected: str
    score: float
    success: bool
    reason: str | None


def make_judge_model() -> AzureOpenAIModel:
    return AzureOpenAIModel(
        model=AZURE_OPENAI_MODEL_NAME,
        deployment_name=AZURE_OPENAI_DEPLOYMENT_NAME,
        api_key=AZURE_OPENAI_API_KEY,
        base_url=AZURE_OPENAI_ENDPOINT,
        api_version=AZURE_OPENAI_API_VERSION,
        cost_per_input_token=AZURE_OPENAI_COST_PER_INPUT_TOKEN,
        cost_per_output_token=AZURE_OPENAI_COST_PER_OUTPUT_TOKEN,
    )

## 6. GitHub MCP Server Definition

In [ ]:
def make_github_mcp_server() -> MCPServer:
    return MCPServer(
        server_name="github",
        transport="stdio",
        available_tools=[
            Tool(
                name="get_issue",
                description="Get a GitHub issue by owner, repository, and issue number.",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "owner": {"type": "string"},
                        "repo": {"type": "string"},
                        "issue_number": {"type": "integer"},
                    },
                    "required": ["owner", "repo", "issue_number"],
                    "additionalProperties": False,
                },
            ),
            Tool(
                name="search_issues",
                description="Search GitHub issues and pull requests.",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "query": {"type": "string"},
                        "per_page": {"type": "integer", "minimum": 1, "maximum": 100},
                    },
                    "required": ["query"],
                    "additionalProperties": False,
                },
            ),
            Tool(
                name="list_commits",
                description="List commits for a GitHub repository.",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "owner": {"type": "string"},
                        "repo": {"type": "string"},
                        "sha": {"type": "string"},
                        "per_page": {"type": "integer", "minimum": 1, "maximum": 100},
                    },
                    "required": ["owner", "repo"],
                    "additionalProperties": False,
                },
            ),
            Tool(
                name="get_repository",
                description="Get metadata for a GitHub repository.",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "owner": {"type": "string"},
                        "repo": {"type": "string"},
                    },
                    "required": ["owner", "repo"],
                    "additionalProperties": False,
                },
            ),
        ],
    )


def make_call_tool_result(result: dict) -> CallToolResult:
    return CallToolResult(
        content=[TextContent(type="text", text=str(result))],
        structuredContent={"result": result},
        isError=False,
    )

## 7. Single-Turn Test Cases

These are used by `MCPUseMetric`.

In [ ]:
def make_single_turn_issue_case(github_server: MCPServer) -> LLMTestCase:
    issue_result = {
        "number": GITHUB_ISSUE_NUMBER,
        "title": "Found a bug",
        "state": "open",
        "url": f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}/issues/{GITHUB_ISSUE_NUMBER}",
    }
    return LLMTestCase(
        input=f"Use GitHub to fetch issue #{GITHUB_ISSUE_NUMBER} from {GITHUB_OWNER}/{GITHUB_REPO} and summarize its status.",
        actual_output=f"I found issue #{GITHUB_ISSUE_NUMBER}, titled '{issue_result['title']}'. It is currently {issue_result['state']}.",
        mcp_servers=[github_server],
        mcp_tools_called=[
            MCPToolCall(
                name="get_issue",
                args={"owner": GITHUB_OWNER, "repo": GITHUB_REPO, "issue_number": GITHUB_ISSUE_NUMBER},
                result=make_call_tool_result(issue_result),
            )
        ],
    )


def make_single_turn_commits_case(github_server: MCPServer) -> LLMTestCase:
    commits_result = {
        "commits": [
            {"sha": "7fd1a60b01f91b314f5994ef42bb1a5d4d5e9d2d", "message": "Merge pull request #6 from Spaceghost/patch-1"},
            {"sha": "762941318ee16e59dabbacb1b4049eec22f0d303", "message": "New line at end of file."},
        ]
    }
    return LLMTestCase(
        input=f"Use GitHub to list the two latest commits in {GITHUB_OWNER}/{GITHUB_REPO}.",
        actual_output="The latest commits include 7fd1a60b01f91b314f5994ef42bb1a5d4d5e9d2d and 762941318ee16e59dabbacb1b4049eec22f0d303.",
        mcp_servers=[github_server],
        mcp_tools_called=[
            MCPToolCall(
                name="list_commits",
                args={"owner": GITHUB_OWNER, "repo": GITHUB_REPO, "per_page": 2},
                result=make_call_tool_result(commits_result),
            )
        ],
    )


def make_single_turn_negative_case(github_server: MCPServer) -> LLMTestCase:
    wrong_result = {
        "commits": [
            {"sha": "badc0ffee0000000000000000000000000000000", "message": "Unrelated commit from the wrong repository."}
        ]
    }
    return LLMTestCase(
        input=f"Use GitHub to fetch issue #{GITHUB_ISSUE_NUMBER} from {GITHUB_OWNER}/{GITHUB_REPO} and summarize its status.",
        actual_output="I could not find that issue, but the repository has one recent commit.",
        mcp_servers=[github_server],
        mcp_tools_called=[
            MCPToolCall(
                name="list_commits",
                args={"owner": "wrong-owner", "repo": "wrong-repo", "per_page": 1},
                result=make_call_tool_result(wrong_result),
            )
        ],
    )

## 8. Multi-Turn Test Cases

These are used by `MultiTurnMCPUseMetric` and `MCPTaskCompletionMetric`.

In [ ]:
def make_multi_turn_issue_case(github_server: MCPServer) -> ConversationalTestCase:
    search_result = {"total_count": 1, "items": [{"number": GITHUB_ISSUE_NUMBER, "title": "Found a bug", "state": "open"}]}
    issue_result = {
        "number": GITHUB_ISSUE_NUMBER,
        "title": "Found a bug",
        "state": "open",
        "body": "There is a reproducible problem in the sample project.",
    }
    return ConversationalTestCase(
        name="GitHub MCP multi-turn issue lookup",
        scenario="A user asks an assistant to find and summarize a GitHub issue.",
        expected_outcome="The assistant should use GitHub MCP tools with correct repository arguments, identify the relevant issue, and summarize its status.",
        mcp_servers=[github_server],
        turns=[
            Turn(role="user", content=f"Find open bug issues in {GITHUB_OWNER}/{GITHUB_REPO} and tell me which one looks most relevant."),
            Turn(
                role="assistant",
                content="I will search the repository issues.",
                mcp_tools_called=[
                    MCPToolCall(
                        name="search_issues",
                        args={"query": f"repo:{GITHUB_OWNER}/{GITHUB_REPO} is:issue is:open bug", "per_page": 5},
                        result=make_call_tool_result(search_result),
                    )
                ],
            ),
            Turn(role="assistant", content=f"The most relevant open bug appears to be #{GITHUB_ISSUE_NUMBER}: Found a bug."),
            Turn(role="user", content="Fetch that issue and summarize the current status."),
            Turn(
                role="assistant",
                content="I will fetch the issue details.",
                mcp_tools_called=[
                    MCPToolCall(
                        name="get_issue",
                        args={"owner": GITHUB_OWNER, "repo": GITHUB_REPO, "issue_number": GITHUB_ISSUE_NUMBER},
                        result=make_call_tool_result(issue_result),
                    )
                ],
            ),
            Turn(role="assistant", content=f"Issue #{GITHUB_ISSUE_NUMBER}, 'Found a bug', is open. The body says the problem is reproducible in the sample project."),
        ],
    )


def make_multi_turn_commits_case(github_server: MCPServer) -> ConversationalTestCase:
    repo_result = {"name": GITHUB_REPO, "owner": GITHUB_OWNER, "default_branch": "master", "visibility": "public"}
    commits_result = {
        "commits": [
            {"sha": "7fd1a60b01f91b314f5994ef42bb1a5d4d5e9d2d", "message": "Merge pull request #6 from Spaceghost/patch-1"},
            {"sha": "762941318ee16e59dabbacb1b4049eec22f0d303", "message": "New line at end of file."},
        ]
    }
    return ConversationalTestCase(
        name="GitHub MCP multi-turn commit lookup",
        scenario="A user asks an assistant to inspect a GitHub repository and summarize commits.",
        expected_outcome="The assistant should use GitHub MCP tools with correct repository arguments, identify the default branch, and summarize recent commits.",
        mcp_servers=[github_server],
        turns=[
            Turn(role="user", content=f"Check {GITHUB_OWNER}/{GITHUB_REPO} and tell me the default branch before listing two recent commits."),
            Turn(
                role="assistant",
                content="I will inspect the repository metadata first.",
                mcp_tools_called=[
                    MCPToolCall(
                        name="get_repository",
                        args={"owner": GITHUB_OWNER, "repo": GITHUB_REPO},
                        result=make_call_tool_result(repo_result),
                    )
                ],
            ),
            Turn(role="assistant", content="The repository is public and its default branch is master."),
            Turn(role="user", content="Now list two recent commits from that repository."),
            Turn(
                role="assistant",
                content="I will retrieve recent commits.",
                mcp_tools_called=[
                    MCPToolCall(
                        name="list_commits",
                        args={"owner": GITHUB_OWNER, "repo": GITHUB_REPO, "sha": "master", "per_page": 2},
                        result=make_call_tool_result(commits_result),
                    )
                ],
            ),
            Turn(role="assistant", content="The default branch is master. Two recent commits are 7fd1a60b01f91b314f5994ef42bb1a5d4d5e9d2d and 762941318ee16e59dabbacb1b4049eec22f0d303."),
        ],
    )


def make_multi_turn_negative_case(github_server: MCPServer) -> ConversationalTestCase:
    wrong_result = {
        "commits": [
            {"sha": "badc0ffee0000000000000000000000000000000", "message": "Unrelated commit from the wrong repository."}
        ]
    }
    return ConversationalTestCase(
        name="GitHub MCP multi-turn negative issue lookup",
        scenario="A user asks an assistant to find and summarize a GitHub issue.",
        expected_outcome="The assistant should call the issue lookup tool with the requested repository and issue number, then summarize the issue status.",
        mcp_servers=[github_server],
        turns=[
            Turn(role="user", content=f"Fetch issue #{GITHUB_ISSUE_NUMBER} from {GITHUB_OWNER}/{GITHUB_REPO} and summarize whether it is open."),
            Turn(
                role="assistant",
                content="I will check recent repository commits.",
                mcp_tools_called=[
                    MCPToolCall(
                        name="list_commits",
                        args={"owner": "wrong-owner", "repo": "wrong-repo", "per_page": 1},
                        result=make_call_tool_result(wrong_result),
                    )
                ],
            ),
            Turn(role="assistant", content="I did not retrieve the issue. I found an unrelated commit instead, so I cannot summarize the issue status."),
        ],
    )

## 9. Build Shared Cases

In [ ]:
judge_model = make_judge_model()
github_server = make_github_mcp_server()

single_turn_cases = [
    ("MCPUse-P1", "Positive", make_single_turn_issue_case(github_server)),
    ("MCPUse-P2", "Positive", make_single_turn_commits_case(github_server)),
    ("MCPUse-N1", "Negative", make_single_turn_negative_case(github_server)),
]

multi_turn_cases = [
    ("P1", "Positive", make_multi_turn_issue_case(github_server)),
    ("P2", "Positive", make_multi_turn_commits_case(github_server)),
    ("N1", "Negative", make_multi_turn_negative_case(github_server)),
]

## 10. Evaluation Helpers

In [ ]:
def truncate_text(value: str | None, max_length: int = 140) -> str:
    if not value:
        return ""
    normalized = " ".join(str(value).split())
    if len(normalized) <= max_length:
        return normalized
    return normalized[: max_length - 3] + "..."


def measure_metric(case_id: str, expected: str, metric, test_case) -> MetricResult:
    score = metric.measure(test_case)
    return MetricResult(
        case_id=case_id,
        metric=metric.__name__,
        expected=expected,
        score=score,
        success=metric.is_successful(),
        reason=getattr(metric, "reason", None),
    )


def results_to_dataframe(results: list[MetricResult]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "Case": result.case_id,
                "Metric": result.metric,
                "Expected": result.expected,
                "Score": round(result.score, 3),
                "Success": result.success,
                "Reason": truncate_text(result.reason),
            }
            for result in results
        ]
    )


def run_metric_group(metric_factory: Callable[[], object], case_specs: list[tuple[str, str, object]]) -> pd.DataFrame:
    results = []
    for case_id, expected, test_case in case_specs:
        metric = metric_factory()
        results.append(measure_metric(case_id, expected, metric, test_case))
    return results_to_dataframe(results)

## 11. MCPUseMetric Results

Single-turn MCP tool correctness and argument generation.

In [ ]:
mcp_use_results = run_metric_group(
    lambda: MCPUseMetric(
        model=judge_model,
        threshold=0.5,
        include_reason=True,
        async_mode=False,
        verbose_mode=True,
    ),
    single_turn_cases,
)

mcp_use_results

## 12. MultiTurnMCPUseMetric Results

Multi-turn MCP tool correctness and argument generation.

In [ ]:
multi_turn_mcp_use_results = run_metric_group(
    lambda: MultiTurnMCPUseMetric(
        model=judge_model,
        threshold=0.5,
        include_reason=True,
        async_mode=False,
        verbose_mode=True,
    ),
    [(f"MultiTurnUse-{case_id}", expected, test_case) for case_id, expected, test_case in multi_turn_cases],
)

multi_turn_mcp_use_results

## 13. MCPTaskCompletionMetric Results

Multi-turn task completion and efficiency.

In [ ]:
mcp_task_completion_results = run_metric_group(
    lambda: MCPTaskCompletionMetric(
        model=judge_model,
        threshold=0.5,
        include_reason=True,
        async_mode=False,
        verbose_mode=True,
    ),
    [(f"TaskCompletion-{case_id}", expected, test_case) for case_id, expected, test_case in multi_turn_cases],
)

mcp_task_completion_results

## 14. Optional Combined Report

In [ ]:
combined_results = pd.concat(
    [mcp_use_results, multi_turn_mcp_use_results, mcp_task_completion_results],
    ignore_index=True,
)

combined_results